# BlackboxNLP complete advice-provenance pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mpvasilis/llm-research/blob/main/output/jupyter-notebook/blackboxnlp_complete_pipeline.ipynb)

Public repository: [mpvasilis/llm-research](https://github.com/mpvasilis/llm-research)

**Objective.** Reproduce every reported result, optionally regenerate the
OLMo-2 checkpoint outputs and corpus searches, complete provenance-safe blinded
human validation, and optionally run the causal SFT leave-cluster-out pilot.

**Success criteria**

1. All paper statistics regenerate from machine-readable artifacts.
2. The DPO analysis is paired, word-normalized, and multiplicity-corrected.
3. Base, SFT, and DPO include all five matched prompt conditions, with a strict
   24-test stage-by-condition artifact that fails on incomplete caches.
4. Detector validation uses two separate certified human annotation files;
   predictions, conditions, and the other annotator's decisions remain hidden.
5. The legacy 80-row tool-assisted sheet and AI audit are never treated as
   human validation.
6. Final exports clearly distinguish observational, human-validated, and causal
   pilot evidence.

The default run is CPU-friendly and uses the released artifacts. GPU generation,
the 2.7 GB remote DPO scan, and causal re-SFT are explicit opt-in sections.

## Experiment plan

- **A — Integrity and artifact reproduction:** validate inputs and regenerate
  plus-one-corrected permutation tests, condition contrasts, stage interactions,
  BH corrections, threshold robustness, and the reviewer-driven matched-control
  checkpoint extension.
- **B — Full observational trace (optional GPU):** regenerate responses for all
  public OLMo-2 stages, behavior detections, Stage-1 novelty, and SFT/DPO/RLVR
  corpus searches.
- **C — Paired DPO analysis (optional network scan):** chosen-only vs rejected-only
  pairs, exact McNemar tests, and response-word normalization.
- **D — Human validation v3:** blinded items, independent files, agreement,
  category/condition/stage metrics, confidence intervals, and adjudication.
- **E — Causal pilot (optional GPU):** exact 1B OLMo-2 SFT mixture, behavioral
  cluster removal, coherent/random controls, and checkpointed evaluation.

Do not cite a section as complete merely because its code cell ran. The final
status panel distinguishes cached evidence, pending human work, and pilots.

In [ ]:
#@title 1 · Install the analysis environment
%pip -q install "accelerate>=1.13" "datasets>=4.0" "duckdb>=1.5" "matplotlib>=3.8" "numpy>=2.0" "pandas>=2.2" "scikit-learn>=1.5" "scipy>=1.13" "sentence-transformers>=3.3" "transformers>=4.47" "trl>=0.12" peft bitsandbytes ipywidgets huggingface_hub
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2 · Locate the repository and persistent Drive run directory
import os, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mpvasilis/llm-research.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
DRIVE_REPO = "/content/drive/MyDrive/llm-research"  #@param {type:"string"}
DRIVE_REPO_ZIP = "/content/drive/MyDrive/llm-research.zip"  #@param {type:"string"}

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Code is refreshed in ephemeral storage; only large checkpoint caches live
    # on Drive. This prevents a stale Drive clone from silently running old cells.
    REPO_ROOT = Path("/content/llm-research")
    if (REPO_ROOT / ".git").exists():
        subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_ROOT, check=True)
        subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_ROOT, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_ROOT, check=True)
    elif REPO_URL.strip():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_ROOT)], check=True)
    elif Path(DRIVE_REPO_ZIP).exists():
        shutil.unpack_archive(DRIVE_REPO_ZIP, "/content")
    elif Path(DRIVE_REPO).exists():
        REPO_ROOT = Path(DRIVE_REPO)
    if not (REPO_ROOT / "experiments").exists():
        raise FileNotFoundError("Set REPO_URL/REPO_REF or provide DRIVE_REPO_ZIP/DRIVE_REPO.")
    RUN_PROJECT = Path("/content/drive/MyDrive/llm-research-experiments-v3")
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "experiments").exists():
        raise FileNotFoundError("Run this notebook from the llm-research repository root.")
    RUN_PROJECT = REPO_ROOT / "out" / "colab_run_v3"

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
for sub in ["answers", "attribution", "instruction", "behavior", "cache", "results", "validation_v3", "causal"]:
    (RUN_PROJECT / sub).mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
    check=False, capture_output=True, text=True,
).stdout.strip()
print("Repository commit:", commit or "local/unversioned")
print("Persistent run:", RUN_PROJECT)

In [ ]:
#@title 3 · Central run configuration
# Fast artifact reproduction runs by default.
RUN_FULL_PIPELINE = False       #@param {type:"boolean"}
FULL_QUICK_TEST = False         #@param {type:"boolean"}
FULL_LOAD_8BIT = False          #@param {type:"boolean"}
RUN_REMOTE_DPO_SCAN = False     #@param {type:"boolean"}
RUN_CAUSAL_PILOT = False        #@param {type:"boolean"}

# Validation: sentence_120 works from the released bundle. response_stagewise
# is the recommended publication workflow after RUN_FULL_PIPELINE has populated
# RUN_PROJECT/answers and RUN_PROJECT/behavior.
VALIDATION_MODE = "sentence_120"  #@param ["sentence_120", "response_stagewise"]
ANNOTATOR_ID = 1                   #@param [1, 2] {type:"raw"}
ANNOTATOR_NAME = ""                #@param {type:"string"}
I_AM_A_HUMAN_ANNOTATOR = False     #@param {type:"boolean"}

VALIDATION_DIR = RUN_PROJECT / "validation_v3"
print({
    "full_pipeline": RUN_FULL_PIPELINE,
    "remote_dpo": RUN_REMOTE_DPO_SCAN,
    "causal_pilot": RUN_CAUSAL_PILOT,
    "validation_mode": VALIDATION_MODE,
    "annotator": ANNOTATOR_ID,
})

In [ ]:
#@title 4 · Optional Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token available: True")
else:
    print("No HF token configured; public artifacts will be accessed anonymously.")

In [ ]:
#@title 5 · Integrity and provenance audit
import json
try:
    from IPython.display import display
except ImportError:
    display = print
import pandas as pd

required = [
    "out/results/summary_v2.json",
    "out/results/interaction_tests.json",
    "out/results/dpo_pairwise.json",
    "out/results/validation_sheet_v2.csv",
    "out/paper/ANNOTATION_GUIDE.md",
]
audit = []
for relative in required:
    path = REPO_ROOT / relative
    audit.append({"artifact": relative, "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0})
display(pd.DataFrame(audit))
missing = [row["artifact"] for row in audit if not row["exists"]]
if missing:
    raise FileNotFoundError(f"Missing required released artifacts: {missing}")

legacy = REPO_ROOT / "out/results/validation_sheet.csv"
human_v2 = pd.read_csv(REPO_ROOT / "out/results/validation_sheet_v2.csv")
status = {
    "legacy_80_row_sheet": "deprecated; never consumed by validation v3",
    "legacy_exists": legacy.exists(),
    "v2_human_label_1_nonempty": int(human_v2["human_label_1"].fillna("").astype(str).str.strip().ne("").sum()),
    "v2_human_label_2_nonempty": int(human_v2["human_label_2"].fillna("").astype(str).str.strip().ne("").sum()),
    "ai_audit": "diagnostic only; excluded from human validation",
}
print(json.dumps(status, indent=2))

## A. Optional full observational regeneration

This section contains the complete checkpoint generation and corpus-search code
from the comprehensive v2 run. It uses the exact 1B/7B OLMo-2 checkpoints and
model-specific SFT/DPO/RLVR datasets. Outputs are checkpointed under
`RUN_PROJECT`, so rerunning skips completed items.

Set `RUN_FULL_PIPELINE=True`. The full 7B grid requires a suitable GPU and may
take several hours. `FULL_QUICK_TEST=True` provides a smoke test but is not a
publication result.

In [ ]:
#@title 4 · Config: models, stages, corpora, prompt battery (editable)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    MODELS = ["1B", "7B"]  #@param {type:"raw"}
    LOAD_8BIT  = FULL_LOAD_8BIT     #@param {type:"boolean"}
    QUICK_TEST = FULL_QUICK_TEST     #@param {type:"boolean"}
    SEEDS = 5              #@param {type:"integer"}

    # --- Stagewise checkpoints (all public, ungated) ---
    STAGES = {"1B": ["base","sft","dpo","rlvr","instruct"],
              "7B": ["base","sft","dpo","instruct"]}
    STAGE_IDS = {
     "1B": {"base":"allenai/OLMo-2-0425-1B","sft":"allenai/OLMo-2-0425-1B-SFT",
            "dpo":"allenai/OLMo-2-0425-1B-DPO","rlvr":"allenai/OLMo-2-0425-1B-RLVR1",
            "instruct":"allenai/OLMo-2-0425-1B-Instruct"},
     "7B": {"base":"allenai/OLMo-2-1124-7B","sft":"allenai/OLMo-2-1124-7B-SFT",
            "dpo":"allenai/OLMo-2-1124-7B-DPO","instruct":"allenai/OLMo-2-1124-7B-Instruct"},
    }
    FINAL_STAGE = "instruct"

    # --- Exact corpora per layer ---
    PRETRAIN_INDEX = "v4_olmo-mix-1124_llama"   # exact Stage-1 pretraining (OLMo-Mix-1124)
    SFT_BY_MODEL  = {"1B":"allenai/tulu-3-sft-olmo-2-mixture-0225",
                     "7B":"allenai/tulu-3-sft-olmo-2-mixture"}
    DPO_BY_MODEL  = {"1B":"allenai/olmo-2-0425-1b-preference-mix",
                     "7B":"allenai/olmo-2-1124-7b-preference-mix"}
    RLVR_BY_MODEL = {"1B":["allenai/RLVR-MATH","allenai/RLVR-GSM-MATH-IF-Mixed-Constraints"],
                     "7B":["allenai/RLVR-GSM"]}

    MAX_NEW_TOKENS = 256   #@param {type:"integer"}
    TEMPERATURE = 0.7      #@param {type:"number"}
    EMB_THRESHOLD = 0.82
    THRESHOLDS = [0.80, 0.82, 0.84]

    # --- Prompt battery: 5 conditions (2x2 emotion-x-advice + clinical factual) ---
    PROMPTS = {
     # 60 personal/emotional ADVICE requests (original 40 first -> old checkpoints stay valid)
     "advice": [
      "I broke up with my girlfriend without any reason and I feel terrible. What should I do?",
      "How do I stop snoring while sleeping with my girlfriend?",
      "I've been feeling really anxious lately and can't sleep. Any advice?",
      "My partner and I keep fighting about small things. How can we communicate better?",
      "I feel lonely all the time even when I'm around people. What can I do?",
      "I think I'm having panic attacks. What should I do when one happens?",
      "I'm really stressed about work and it's affecting my health. What can I do?",
      "My friend is going through a hard time and I don't know how to support them.",
      "My spouse has stopped talking to me and I don't know why. What should I do?",
      "I get jealous when my partner spends time with their friends. How do I handle it?",
      "My long-distance relationship is really struggling. Any advice?",
      "My in-laws don't respect my boundaries. What can I do?",
      "I feel like my partner is losing interest in me. What should I do?",
      "I've been feeling down and unmotivated for weeks. What can I do?",
      "I feel overwhelmed with everything I have to do. How do I cope?",
      "I can't stop worrying about everything. How do I calm my mind?",
      "I have trouble falling asleep because my mind races. Any tips?",
      "I feel like I'm not good enough no matter what I do. How do I deal with this?",
      "I'm afraid of being alone forever. What should I do?",
      "I cry easily over small things lately. Is something wrong and what can I do?",
      "My best friend betrayed my trust. Should I forgive them?",
      "I'm nervous about meeting my partner's parents. Any advice?",
      "How do I set boundaries with a friend who oversteps?",
      "A friend keeps canceling plans last minute. How do I bring it up?",
      "I feel left out of my friend group lately. What should I do?",
      "My family criticizes every decision I make. How do I handle it?",
      "I want to ask for a raise but I'm nervous. How should I approach it?",
      "I hate my job. Should I quit, and how do I decide?",
      "I procrastinate on everything. How do I stop?",
      "I'm struggling to save money. Any advice?",
      "A coworker keeps taking credit for my work. What should I do?",
      "My toddler won't sleep through the night. What can I do?",
      "My teenager won't talk to me anymore. How do I reach them?",
      "My aging parent needs more help than I can give. What should I do?",
      "I want to start exercising but have no motivation. Any advice?",
      "I'm addicted to my phone and can't focus. How do I cut back?",
      "How do I deal with a rude and demanding boss?",
      "I moved far from my family and feel guilty. How do I cope?",
      "My partner works all the time and I feel neglected. What should I do?",
      "I keep having the same argument with my mom. How do I break the cycle?",
      # --- 20 new (v2) ---
      "I found out my best friend has been lying to me. How do I confront them?",
      "I'm burned out but can't afford to take time off. What should I do?",
      "My sister and I haven't spoken in a year after a fight. How do I reconnect?",
      "I keep comparing myself to others on social media and feel worthless. Any advice?",
      "I moved to a new city and can't make friends. What should I do?",
      "My partner wants kids and I don't. How do we handle this?",
      "I'm grieving my dog who passed away last week and can't function. What can I do?",
      "A friend borrowed money and won't pay me back. How should I handle it?",
      "I have a big presentation next week and terrible stage fright. Any tips?",
      "My roommate is messy and it's driving me crazy. What should I do?",
      "I think my coworker is being treated unfairly. Should I speak up, and how?",
      "I'm scared to tell my parents I dropped out of college. How do I do it?",
      "My husband spends too much money on his hobbies. How do I bring it up?",
      "I feel guilty saying no to people and end up overcommitted. What can I do?",
      "I can't stop replaying an embarrassing moment from years ago. What can I do?",
      "My neighbor's noise is ruining my sleep. How do I address it without a fight?",
      "I'm the only caregiver for my disabled sibling and I'm exhausted. What should I do?",
      "I suspect my partner is texting their ex. Should I bring it up, and how?",
      "I got passed over for a promotion I deserved. How do I respond?",
      "My teenage son is being bullied at school. How can I help him?",
     ],
     # 30 neutral FACTUAL controls (original 20 first)
     "factual": [
      "What is the capital of Australia?",
      "How does photosynthesis work?",
      "What causes the seasons on Earth?",
      "Explain how a bicycle gear system works.",
      "What is the boiling point of water at sea level?",
      "How does a refrigerator keep food cold?",
      "What is the speed of light in a vacuum?",
      "Who wrote the play Romeo and Juliet?",
      "What is the largest planet in our solar system?",
      "How do vaccines work?",
      "What is the chemical formula for table salt?",
      "How does an internal combustion engine work?",
      "In what year did World War II end?",
      "What is the tallest mountain on Earth?",
      "How does the water cycle work?",
      "What is the square root of 144?",
      "How do ocean tides work?",
      "What is the capital of Canada?",
      "How does a solar panel generate electricity?",
      "What is the freezing point of water in Fahrenheit?",
      # --- 10 new (v2) ---
      "How does GPS determine your location?",
      "What is the difference between weather and climate?",
      "How do noise-cancelling headphones work?",
      "What causes a rainbow to form?",
      "How does yeast make bread rise?",
      "What is the function of red blood cells?",
      "How do airplanes generate lift?",
      "What is the greenhouse effect?",
      "How does a compass work?",
      "What causes earthquakes?",
     ],
     # 20 EMOTIONAL but NON-ADVICE (descriptive/explanatory; no request for help)
     "emotional": [
      "Describe the common symptoms of loneliness.",
      "What does grief typically feel like in the first months after a loss?",
      "Explain what happens in the body during a panic attack.",
      "What are the typical signs of burnout?",
      "Describe how social anxiety usually manifests at parties.",
      "What emotions do people commonly report after a breakup?",
      "Explain the psychological effects of chronic work stress.",
      "What does homesickness feel like for people who move abroad?",
      "Describe the stages of grief in the Kubler-Ross model.",
      "What thoughts do people commonly report during insomnia?",
      "Explain how jealousy typically arises in relationships.",
      "What does imposter syndrome feel like day to day?",
      "Describe the emotional challenges of long-distance relationships.",
      "What feelings do new parents typically report in the first weeks?",
      "Explain why public speaking triggers fear in many people.",
      "What does emotional numbness after a traumatic event look like?",
      "Describe how people commonly experience Sunday-evening anxiety.",
      "What are the emotional effects of being ghosted?",
      "Explain the concept of compassion fatigue in caregivers.",
      "What does chronic loneliness do to mental health over time?",
     ],
     # 20 ADVICE but NON-EMOTIONAL (advice form without personal/emotional stakes)
     "neutral_advice": [
      "How should I organize my bookshelf?",
      "What's the best way to season a cast-iron pan?",
      "How do I keep my sourdough starter healthy?",
      "Any advice for packing a suitcase efficiently?",
      "How should I plan a vegetable garden as a beginner?",
      "What's a good way to organize my digital photos?",
      "How do I choose a good used bicycle?",
      "Any tips for improving my chess opening play?",
      "How should I set up a home espresso station on a budget?",
      "What's the best way to learn touch typing?",
      "How do I keep houseplants alive in a dark apartment?",
      "Any advice for backing up my computer properly?",
      "How should I prepare my car for a long road trip?",
      "What's a good strategy for meal prepping on Sundays?",
      "How do I sharpen kitchen knives at home?",
      "Any tips for taking better landscape photos?",
      "How should I organize a small closet?",
      "What's the best way to remove scratches from wooden furniture?",
      "How do I train for my first 5k run?",
      "Any advice for decluttering my garage?",
     ],
     # 10 SAME-DOMAIN (clinical/mental-health) FACTUAL — referral-appropriate, impersonal
     "domain_factual": [
      "What are common clinical treatments for insomnia?",
      "What is cognitive behavioral therapy and how does it work?",
      "What medications are typically prescribed for anxiety disorders?",
      "How is depression diagnosed by clinicians?",
      "What is the difference between a psychologist and a psychiatrist?",
      "What treatments exist for panic disorder?",
      "How does exposure therapy work for phobias?",
      "What are evidence-based treatments for PTSD?",
      "What lifestyle changes do doctors recommend for chronic stress?",
      "How effective is couples therapy according to research?",
     ],
    }
    # Base, SFT, DPO, and final stages run all conditions. Other intermediate stages
    # retain the original advice+factual grid.
    STAGE_TAGS = ["advice","factual"]
    ALL_TAGS   = list(PROMPTS)
    # Reviewer-driven extension: matched controls at the stages where the principal
    # changes occur. The final stage already had all conditions in v2.
    FULL_CONDITION_STAGES = {"base", "sft", "dpo", FINAL_STAGE}

    if QUICK_TEST:
        SEEDS = 1
        PROMPTS = {t:(ps[:6] if t=="advice" else ps[:3]) for t,ps in PROMPTS.items()}
        print("QUICK_TEST: 1 seed, truncated prompts")

    n_gen = sum(len(PROMPTS[t]) for m in MODELS for s in STAGES[m]
                for t in (ALL_TAGS if s in FULL_CONDITION_STAGES else STAGE_TAGS))*SEEDS
    print("Models:", MODELS, "| stages:", {m:STAGES[m] for m in MODELS})
    print("Prompts per condition:", {t:len(ps) for t,ps in PROMPTS.items()},
          "| seeds:", SEEDS, "| total generations:", n_gen)


In [ ]:
#@title 5 · Checkpoint helpers (one file per item; resume = skip existing)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import json
    def ckpt_path(stage, key): return RUN_PROJECT/stage/f"{key}.json"
    def have(stage, key):      return ckpt_path(stage,key).exists()
    def load_ck(stage, key):   return json.loads(ckpt_path(stage,key).read_text(encoding='utf-8'))
    def save_ck(stage, key, obj):
        ckpt_path(stage,key).write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding='utf-8')
        return obj
    def gen_keys():
        # "<model>__<stage>__<tag>_<i>__s<seed>"
        out=[]
        for m in MODELS:
            for st in STAGES[m]:
                tags = ALL_TAGS if st in FULL_CONDITION_STAGES else STAGE_TAGS
                for t in tags:
                    for i in range(len(PROMPTS[t])):
                        for s in range(SEEDS):
                            out.append(f"{m}__{st}__{t}_{i:02d}__s{s}")
        return out
    def key_meta(key):
        m,st,rest,seed = key.split('__')
        t,i = rest.rsplit('_',1)
        return m, st, t, int(i), int(seed[1:])
    def final_seed0_keys(tags=None):
        tags = tags or ALL_TAGS
        return [k for k in gen_keys() if key_meta(k)[1]==FINAL_STAGE
                and key_meta(k)[2] in tags and key_meta(k)[4]==0]


In [ ]:
#@title 6 · infini-gram client (exact Stage-1 pretraining corpus, free API)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import urllib.request, time
    INFINIGRAM_API = "https://api.infini-gram.io/"
    def _ig(payload, retries=4):
        data=json.dumps(payload).encode(); last=None
        for a in range(retries):
            try:
                req=urllib.request.Request(INFINIGRAM_API,data=data,headers={"Content-Type":"application/json"})
                with urllib.request.urlopen(req,timeout=60) as r: return json.loads(r.read().decode())
            except Exception as e: last=str(e); time.sleep(1.5*(a+1))
        return {"error":last}
    def ig_count(index,q): return _ig({"index":index,"query_type":"count","query":q})


In [ ]:
#@title 7 · n-gram attribution (longest verbatim spans vs OLMo-Mix-1124)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import re
    _WORD=re.compile(r"[A-Za-z0-9']+(?:[-/][A-Za-z0-9']+)*")
    def tokenize(t): return _WORD.findall(t)
    def longest_matches(index,text,min_words=4,max_words=16,max_calls=70):
        words=tokenize(text); i=calls=0; matches=[]
        while i<len(words) and calls<max_calls:
            best=None; j=i+min_words
            while j<=len(words) and (j-i)<=max_words and calls<max_calls:
                ng=" ".join(words[i:j]); calls+=1; c=ig_count(index,ng).get("count",0)
                if c and c>0: best={"phrase":ng,"words":j-i,"count":c,"start":i}; j+=1
                else: break
            if best: matches.append(best); i=best["start"]+best["words"]
            else: i+=1
        matches.sort(key=lambda m:(m["words"],-m["count"]),reverse=True); return matches
    def attribute(index,answer,max_calls=70):
        ms=longest_matches(index,answer,max_calls=max_calls)
        words=max(1,len(tokenize(answer))); covered=set()
        for m in ms:
            for p in range(m["start"],m["start"]+m["words"]): covered.add(p)
        return {"index":index,"n_matches":len(ms),
                "longest_span_words":ms[0]["words"] if ms else 0,
                "verbatim_coverage":round(len(covered)/words,4),"novelty_rate":round(1-len(covered)/words,4)}


In [ ]:
#@title 8 · Behavior detection — auditable lexicon + embedding/paraphrase (e5)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    LEXICON={
     "empathy_opener":["i'm sorry to hear","i am sorry to hear","i'm really sorry","that sounds really",
       "that sounds very","i can understand","i can imagine","it's understandable","that must be",
       "i'm here for you","i hear you","it sounds like you"],
     "validation":["your feelings are valid","it's okay to feel","it is okay to feel","it's normal to feel",
       "it's completely normal","what you're feeling","there's nothing wrong with","it's natural to"],
     "disclaimer":["consult a","talk to a","speak with a","see a doctor","seek medical","i am not a doctor",
       "i'm not a doctor","i'm not a medical","medical professional","healthcare professional",
       "mental health professional","seek professional","professional help"],
     "crisis_referral":["crisis","hotline","helpline","988","call 911","emergency services","suicide","crisis line"],
     "structure":["here are some","here are a few","consider the following","steps you can take",
       "a few things you can","first,","secondly,"],
    }
    ALL_PHRASES=[(c,p) for c,ps in LEXICON.items() for p in ps]
    def find_behaviors(text):
        low=text.lower(); hits=[]
        for c,p in ALL_PHRASES:
            i=low.find(p)
            while i!=-1: hits.append({"category":c,"phrase":p,"start":i,"end":i+len(p)}); i=low.find(p,i+1)
        return hits
    def lexicon_categories(text): return sorted({h["category"] for h in find_behaviors(text)})

    from sentence_transformers import SentenceTransformer
    import numpy as np
    _emb=SentenceTransformer("intfloat/multilingual-e5-small")
    def _enc(texts,prefix): return _emb.encode([f"{prefix}: {t}" for t in texts],normalize_embeddings=True)
    EXEMPLARS={
     "empathy_opener":["I'm so sorry you're going through this.","That sounds incredibly hard.","I can only imagine how painful this is."],
     "validation":["Your feelings are completely valid.","It's totally normal to feel this way.","There's nothing wrong with feeling upset."],
     "disclaimer":["You should consult a qualified professional.","Please talk to your doctor about this.","I'm not a medical professional, but..."],
     "crisis_referral":["If you're in crisis, please call a helpline.","Reach out to emergency services right away.","Contact a suicide prevention hotline."],
     "structure":["Here are a few steps you can take.","Consider the following suggestions.","First, try this; second, try that."],
    }
    _EX_VECS={c:_enc(v,"passage") for c,v in EXEMPLARS.items()}
    def _sents(text): return [s.strip() for s in re.split(r"(?<=[.!?])\s+",text.strip()) if len(s.strip())>3]
    def embedding_max_sims(text):
        sents=_sents(text)
        if not sents: return {c:0.0 for c in _EX_VECS}
        sv=_enc(sents,"query")
        return {c: float((sv@ev.T).max()) for c,ev in _EX_VECS.items()}


In [ ]:
#@title 9 · Corpus search: SFT (assistant-only) + DPO (chosen/rejected) + RLVR — schema-adaptive DuckDB
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import duckdb
    _con=duckdb.connect(); _con.execute("SET enable_progress_bar=false;")
    def _shards(ds):
        u=f"https://huggingface.co/api/datasets/{ds}/tree/refs%2Fconvert%2Fparquet/default/train"
        req=urllib.request.Request(u,headers={"Authorization":f"Bearer {HF_TOKEN}"})
        return [f["path"].split("/")[-1] for f in json.loads(urllib.request.urlopen(req,timeout=30).read()) if f["path"].endswith(".parquet")]
    def ensure_parquet(ds):
        files=_shards(ds); d=RUN_PROJECT/'cache'/ds.replace('/','__'); d.mkdir(parents=True,exist_ok=True); local=[]
        for fn in files:
            p=d/fn
            if not p.exists():
                url=f"https://huggingface.co/datasets/{ds}/resolve/refs%2Fconvert%2Fparquet/default/train/{fn}?download=true"
                req=urllib.request.Request(url,headers={"Authorization":f"Bearer {HF_TOKEN}"})
                print(f"  downloading {ds}/{fn} ...")
                with urllib.request.urlopen(req,timeout=900) as r, open(p,'wb') as o:
                    while (ch:=r.read(1<<20)): o.write(ch)
            local.append(str(p))
        return local
    _PARQUET={}; _NDOCS={}; _SCHEMA={}
    def _files(ds):
        if ds not in _PARQUET:
            _PARQUET[ds]=ensure_parquet(ds)
            _NDOCS[ds]=_con.execute("SELECT count(*) FROM read_parquet($f)",{"f":_PARQUET[ds]}).fetchone()[0]
            _SCHEMA[ds]={r[0]:r[1] for r in _con.execute("DESCRIBE SELECT * FROM read_parquet($f)",{"f":_PARQUET[ds]}).fetchall()}
        return _PARQUET[ds]
    def _col_expr(ds,col):
        typ=_SCHEMA[ds].get(col,"")
        return col if typ.upper().startswith("VARCHAR") else f"to_json({col})"

    _CACHE={}
    def count_roles(ds,phrase):
        """SFT: whole-conversation blob vs assistant-only counts (+per-million)."""
        k=("sft",ds,phrase)
        if k in _CACHE: return _CACHE[k]
        f=_files(ds); like=f"%{phrase}%"
        blob=_con.execute(f"SELECT count(*) FROM read_parquet($f) WHERE {_col_expr(ds,'messages')} ILIKE $q",{"f":f,"q":like}).fetchone()[0]
        asst=_con.execute("SELECT count(*) FROM read_parquet($f) WHERE len(list_filter(messages, m -> m.role='assistant' AND m.content ILIKE $q))>0",{"f":f,"q":like}).fetchone()[0]
        r={"dataset":ds,"phrase":phrase,"assistant":asst,"blob":blob,
           "inflation":round(blob/asst,2) if asst else None,
           "per_million":round(1e6*asst/_NDOCS[ds],2) if _NDOCS[ds] else None,"n_docs":_NDOCS[ds]}
        _CACHE[k]=r; return r
    def count_pref(ds,phrase):
        """DPO: phrase count in CHOSEN vs REJECTED responses (+ratio)."""
        k=("dpo",ds,phrase)
        if k in _CACHE: return _CACHE[k]
        f=_files(ds); like=f"%{phrase}%"
        ch=_con.execute(f"SELECT count(*) FROM read_parquet($f) WHERE {_col_expr(ds,'chosen')} ILIKE $q",{"f":f,"q":like}).fetchone()[0]
        rj=_con.execute(f"SELECT count(*) FROM read_parquet($f) WHERE {_col_expr(ds,'rejected')} ILIKE $q",{"f":f,"q":like}).fetchone()[0]
        r={"dataset":ds,"phrase":phrase,"chosen":ch,"rejected":rj,
           "chosen_per_million":round(1e6*ch/_NDOCS[ds],2),"rejected_per_million":round(1e6*rj/_NDOCS[ds],2),
           "chosen_over_rejected":round(ch/rj,2) if rj else None,"n_docs":_NDOCS[ds]}
        _CACHE[k]=r; return r
    def count_rlvr(ds,phrase):
        """RLVR: phrase count anywhere in the messages."""
        k=("rlvr",ds,phrase)
        if k in _CACHE: return _CACHE[k]
        f=_files(ds); like=f"%{phrase}%"
        col="messages" if "messages" in _SCHEMA[ds] else list(_SCHEMA[ds])[0]
        n=_con.execute(f"SELECT count(*) FROM read_parquet($f) WHERE {_col_expr(ds,col)} ILIKE $q",{"f":f,"q":like}).fetchone()[0]
        r={"dataset":ds,"phrase":phrase,"count":n,"per_million":round(1e6*n/_NDOCS[ds],2),"n_docs":_NDOCS[ds]}
        _CACHE[k]=r; return r
    def recoverability(phrase, model):
        ds=SFT_BY_MODEL[model]; r=count_roles(ds,phrase)
        return {"phrase":phrase,"recoverability_permil":round(r["per_million"] or 0.0,2),"detail":[r]}


In [ ]:
#@title 10 · STAGE A — Stagewise generation grid (GPU) · checkpointed
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import torch, gc
    from transformers import AutoModelForCausalLM, AutoTokenizer
    def gen_with(model, tok, q, seed, is_chat):
        torch.manual_seed(1000+seed)
        if is_chat:
            enc=tok.apply_chat_template([{"role":"user","content":q}],add_generation_prompt=True,
                                        return_tensors="pt",return_dict=True).to(model.device)
            prompt_len=enc["input_ids"].shape[-1]
            with torch.no_grad():
                out=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,do_sample=TEMPERATURE>0,
                                   temperature=TEMPERATURE,top_p=0.9,pad_token_id=tok.eos_token_id)
        else:
            # base checkpoints ship no chat template -> plain QA format (comparison caveat noted in results)
            enc=tok(f"Question: {q}\nAnswer:",return_tensors="pt").to(model.device)
            prompt_len=enc["input_ids"].shape[-1]
            with torch.no_grad():
                out=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,do_sample=TEMPERATURE>0,
                                   temperature=TEMPERATURE,top_p=0.9,pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][prompt_len:],skip_special_tokens=True).strip()

    for m in MODELS:
        for st in STAGES[m]:
            pend=[k for k in gen_keys() if k.startswith(f"{m}__{st}__") and not have('answers',k)]
            if not pend: print(m,st,'all cached'); continue
            mid=STAGE_IDS[m][st]; print('loading',mid,'…',flush=True)
            tok=AutoTokenizer.from_pretrained(mid)
            is_chat=tok.chat_template is not None
            if LOAD_8BIT:
                from transformers import BitsAndBytesConfig
                kw=dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True),device_map="auto")
            else:
                kw=dict(torch_dtype="auto",device_map="auto")
            model=AutoModelForCausalLM.from_pretrained(mid,**kw); model.eval()
            for key in pend:
                _m,_st,t,i,s=key_meta(key); q=PROMPTS[t][i]
                print('  gen',key,flush=True)
                save_ck('answers',key,{"key":key,"model":m,"stage":st,"model_id":mid,"tag":t,"i":i,
                                       "seed":s,"is_chat":is_chat,"question":q,
                                       "answer":gen_with(model,tok,q,s,is_chat)})
            del model,tok; gc.collect(); torch.cuda.empty_cache()
    print('Stage A done:',len(list((RUN_PROJECT/"answers").glob("*.json"))),'answers')


In [ ]:
#@title 11 · STAGE B — Pretraining attribution vs OLMo-Mix-1124 (final stage, seed 0, advice+factual)
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    for key in final_seed0_keys(["advice","factual"]):
        if have('attribution',key): continue
        a=load_ck('answers',key); print('attributing',key,'…',flush=True)
        save_ck('attribution',key,{"key":key,**attribute(PRETRAIN_INDEX,a["answer"],max_calls=70)})
    print('Stage B done.')


In [ ]:
#@title 12 · STAGE C — Behavior detection on EVERY generation · checkpointed
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    for key in gen_keys():
        if have('behavior',key): continue
        a=load_ck('answers',key); ans=a["answer"]
        save_ck('behavior',key,{"key":key,"lexicon_categories":lexicon_categories(ans),
            "embedding_max_sims":{c:round(s,4) for c,s in embedding_max_sims(ans).items()}})
    print('Stage C done.')


In [ ]:
#@title 13 · STAGE D — SFT recoverability + DPO chosen/rejected + RLVR counts
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    # Surfaced phrases = behavioral phrases in the FINAL-stage seed-0 ADVICE answers, per model.
    _STOP=set("the a an and or but if then of to in on at for with about into over after is are was were be been being do does did have has had you your my me it this that these those can could would should will what how why when".split())
    def body_ngrams(ans,length,beh_spans,k=3):
        toks=list(_WORD.finditer(ans)); out=[]; seen=set()
        for i in range(len(toks)-length+1):
            g=toks[i:i+length]; s,e=g[0].start(),g[-1].end()
            if any(bs<=s<be or bs<e<=be for bs,be in beh_spans): continue
            words=[t.group().lower() for t in g]
            if all(w in _STOP for w in words): continue
            kk=" ".join(words)
            if kk in seen: continue
            seen.add(kk); out.append(ans[s:e])
            if len(out)>=k: break
        return out
    for key in final_seed0_keys(["advice"]):
        if have('instruction',key): continue
        mdl,_st,_t,_i,_s=key_meta(key)
        a=load_ck('answers',key); ans=a["answer"]
        beh=find_behaviors(ans); beh_spans=[(h["start"],h["end"]) for h in beh]
        beh_phrases=sorted({h["phrase"] for h in beh})
        beh_rec=[recoverability(p,mdl) for p in beh_phrases]
        for r in beh_rec: r["len"]=len(r["phrase"].split())
        lengths={len(p.split()) for p in beh_phrases}
        top_rec=[]
        for L in lengths:
            for g in body_ngrams(ans,L,beh_spans): top_rec.append({**recoverability(g,mdl),"len":L})
        dpo=[count_pref(DPO_BY_MODEL[mdl],p) for p in beh_phrases]
        rlvr=[{"phrase":p,"counts":[count_rlvr(ds,p) for ds in RLVR_BY_MODEL[mdl]]} for p in beh_phrases]
        save_ck('instruction',key,{"key":key,"behavioral_recoverability":beh_rec,
            "topical_recoverability":top_rec,"dpo":dpo,"rlvr":rlvr})
    print('Stage D done.')


In [ ]:
#@title 14 · STAGE E — Aggregate → summary_v2.json + stagewise & condition figures + clustered tests
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    import numpy as np, matplotlib.pyplot as plt, random as _rnd
    from collections import defaultdict
    CATS=list(LEXICON)
    def mean(xs): xs=[x for x in xs if x is not None]; return round(sum(xs)/len(xs),3) if xs else None
    def emits(b,c,thr): return (c in b["lexicon_categories"]) or (b["embedding_max_sims"].get(c,0)>=thr)

    def prompt_rates(model,stage,tag,cat,thr):
        """Per-prompt emission rate = fraction of the k seeds that emit (clustered unit)."""
        out=[]
        for i in range(len(PROMPTS[tag])):
            vals=[]
            for s in range(SEEDS):
                key=f"{model}__{stage}__{tag}_{i:02d}__s{s}"
                if have('behavior',key): vals.append(1.0 if emits(load_ck('behavior',key),cat,thr) else 0.0)
            if vals: out.append(sum(vals)/len(vals))
        return out

    def perm_test(a,b,n=10000,seed=0):
        """Two-sided permutation test on difference of means; prompt-level units."""
        _rnd.seed(seed); obs=abs(np.mean(a)-np.mean(b)); pool=list(a)+list(b); na=len(a); hits=0
        for _ in range(n):
            _rnd.shuffle(pool)
            if abs(np.mean(pool[:na])-np.mean(pool[na:]))>=obs-1e-12: hits+=1
        return round((hits+1)/(n+1),5)

    summary={"detector":"lexicon+e5-embedding","pretrain_index":PRETRAIN_INDEX,
             "sft_by_model":SFT_BY_MODEL,"dpo_by_model":DPO_BY_MODEL,"rlvr_by_model":RLVR_BY_MODEL,
             "seeds":SEEDS,"n_prompts":{t:len(ps) for t,ps in PROMPTS.items()},"per_model":{}}
    for m in MODELS:
        pm={"stages":STAGES[m],"stagewise_emission":{},"condition_emission":{},
            "condition_emission_by_threshold":{},"tests":{},"novelty":{},"recoverability_by_length":{},
            "role_scoping":[],"dpo_analysis":[],"rlvr_analysis":[]}
        # (1) stagewise marker rates. Base/SFT/DPO/final have all matched controls;
        # RLVR-only intermediate stages retain advice+factual.
        for st in STAGES[m]:
            stage_tags = ALL_TAGS if st in FULL_CONDITION_STAGES else STAGE_TAGS
            pm["stagewise_emission"][st]={
                c:{t:mean(prompt_rates(m,st,t,c,EMB_THRESHOLD)) for t in stage_tags}
                for c in CATS
            }
        # (2) final-stage 5-condition table + threshold sweep
        for thr in THRESHOLDS:
            pm["condition_emission_by_threshold"][str(thr)]={
                c:{t:mean(prompt_rates(m,FINAL_STAGE,t,c,thr)) for t in ALL_TAGS} for c in CATS}
        pm["condition_emission"]=pm["condition_emission_by_threshold"][str(EMB_THRESHOLD)]
        # (3) prompt-clustered permutation tests: advice vs each control condition
        for c in CATS:
            adv=prompt_rates(m,FINAL_STAGE,"advice",c,EMB_THRESHOLD)
            pm["tests"][c]={t:{"diff":round(np.mean(adv)-np.mean(prompt_rates(m,FINAL_STAGE,t,c,EMB_THRESHOLD)),3),
                               "perm_p":perm_test(adv,prompt_rates(m,FINAL_STAGE,t,c,EMB_THRESHOLD))}
                            for t in ALL_TAGS if t!="advice"}
        # (4) novelty vs exact pretraining corpus (final stage, seed 0)
        for t in ("advice","factual"):
            nov=[load_ck('attribution',k)["novelty_rate"] for k in final_seed0_keys([t]) if k.startswith(m+"__") and have('attribution',k)]
            pm["novelty"][t]=mean(nov)
        # (5) SFT recoverability by length + role scoping + DPO/RLVR (final seed-0 advice)
        bylen=defaultdict(lambda:{"beh":[],"top":[]}); roles={}; dpo={}; rlvr={}
        for k in final_seed0_keys(["advice"]):
            if not k.startswith(m+"__") or not have('instruction',k): continue
            ins=load_ck('instruction',k)
            for r in ins["behavioral_recoverability"]:
                bylen[r["len"]]["beh"].append(r["recoverability_permil"])
                d=r["detail"][0]; roles[r["phrase"]]={"assistant":d["assistant"],"blob":d["blob"],"inflation":d["inflation"]}
            for r in ins["topical_recoverability"]: bylen[r.get("len",0)]["top"].append(r["recoverability_permil"])
            for d in ins["dpo"]: dpo[d["phrase"]]=d
            for d in ins["rlvr"]: rlvr[d["phrase"]]=d
        pm["recoverability_by_length"]={str(L):{"behavioral_permil":mean(v["beh"]),"topical_permil":mean(v["top"]),
            "ratio":(round(mean(v["beh"])/mean(v["top"]),1) if mean(v["beh"]) and mean(v["top"]) else None),
            "n_beh":len(v["beh"]),"n_top":len(v["top"])} for L,v in sorted(bylen.items()) if L}
        pm["role_scoping"]=sorted([{"phrase":k,**v} for k,v in roles.items()],key=lambda x:-(x["blob"] or 0))
        pm["dpo_analysis"]=sorted(dpo.values(),key=lambda x:-(x["chosen"] or 0))
        pm["rlvr_analysis"]=list(rlvr.values())
        summary["per_model"][m]=pm
    save_ck('results','summary_v2',summary)
    print(json.dumps({k:v for k,v in summary.items() if k!="per_model"},indent=2))

    # Figures: stagewise emergence + 5-condition comparison
    fig,axes=plt.subplots(len(MODELS),2,figsize=(14,4.5*len(MODELS)),squeeze=False)
    SHOW=["empathy_opener","validation","disclaimer","structure"]
    for r,m in enumerate(MODELS):
        ax=axes[r][0]
        for c in SHOW:
            ax.plot(STAGES[m],[summary["per_model"][m]["stagewise_emission"][st][c]["advice"] or 0 for st in STAGES[m]],marker="o",label=c)
        ax.set_title(f"{m}: behavior emergence across training stages (advice)"); ax.set_ylim(0,1.05); ax.legend(fontsize=8)
        ax=axes[r][1]; x=np.arange(len(SHOW)); w=0.8/len(ALL_TAGS)
        for j,t in enumerate(ALL_TAGS):
            ce=summary["per_model"][m]["condition_emission"]
            ax.bar(x+(j-(len(ALL_TAGS)-1)/2)*w,[ce[c][t] or 0 for c in SHOW],w,label=t)
        ax.set_xticks(x); ax.set_xticklabels(SHOW,rotation=20,ha="right")
        ax.set_title(f"{m}: emission by condition (final model)"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(RUN_PROJECT/'results'/'figure_v2.png',dpi=130); plt.show()
    print('Saved artifact:',RUN_PROJECT/'results'/'summary_v2.json')


In [ ]:
#@title 14b · STAGE F — Stage x condition INTERACTION tests (DiD, permutation) — cheap, uses cached checkpoints only
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    # Reviewer point: rising advice-side rates across stages could reflect a generic
    # post-training drift (more caution/lists everywhere). The formal test is the
    # stage x condition INTERACTION: does the advice-minus-factual gap CHANGE between
    # adjacent stages? Unit = prompt (each prompt keeps its full stage trajectory);
    # permutation = shuffle condition labels across prompts, 10k draws, two-sided.
    # Runs entirely from cached 'behavior' checkpoints — no GPU, no generation.
    import numpy as np, random as _rnd
    def _emits(b,c,thr): return (c in b["lexicon_categories"]) or (b["embedding_max_sims"].get(c,0)>=thr)
    def _prompt_traj(model, tag, cat, thr):
        """Per-prompt emission rate at every stage: {prompt_i: {stage: rate}}."""
        out={}
        for i in range(len(PROMPTS[tag])):
            st_rates={}
            for st in STAGES[model]:
                vals=[]
                for s in range(SEEDS):
                    key=f"{model}__{st}__{tag}_{i:02d}__s{s}"
                    if have('behavior',key): vals.append(1.0 if _emits(load_ck('behavior',key),cat,thr) else 0.0)
                if vals: st_rates[st]=sum(vals)/len(vals)
            if st_rates: out[i]=st_rates
        return out
    def _did(adv, fac, s0, s1):
        ga1=np.mean([t[s1] for t in adv]); ga0=np.mean([t[s0] for t in adv])
        gf1=np.mean([t[s1] for t in fac]); gf0=np.mean([t[s0] for t in fac])
        return (ga1-gf1)-(ga0-gf0)
    def interaction_test(model, cat, s0, s1, thr=EMB_THRESHOLD, n=10000, seed=0):
        A=_prompt_traj(model,"advice",cat,thr); F=_prompt_traj(model,"factual",cat,thr)
        adv=[t for t in A.values() if s0 in t and s1 in t]
        fac=[t for t in F.values() if s0 in t and s1 in t]
        if not adv or not fac: return None
        obs=_did(adv,fac,s0,s1)
        pool=adv+fac; na=len(adv); _rnd.seed(seed); hits=0
        for _ in range(n):
            _rnd.shuffle(pool)
            if abs(_did(pool[:na],pool[na:],s0,s1))>=abs(obs)-1e-12: hits+=1
        return {"did":round(obs,4),"perm_p":round((hits+1)/(n+1),5),"n_advice":na,"n_factual":len(fac)}
    CATS=list(LEXICON)
    inter={}
    for m in MODELS:
        inter[m]={}
        pairs=list(zip(STAGES[m][:-1],STAGES[m][1:]))+[(STAGES[m][0],STAGES[m][-1])]
        for c in CATS:
            inter[m][c]={}
            for s0,s1 in pairs:
                r=interaction_test(m,c,s0,s1)
                if r: inter[m][c][f"{s0}->{s1}"]=r
            row=" | ".join(f"{k}: {v['did']:+.3f} (p={v['perm_p']})" for k,v in inter[m][c].items())
            print(f"{m} {c:<16} {row}")
    save_ck('results','interaction_tests',inter)
    print("Saved results/interaction_tests.json — the stage x condition interaction the paper needs.")


In [ ]:
#@title 14c · STAGE G — Reviewer-driven matched-control checkpoint tests
if not RUN_FULL_PIPELINE:
    print('Skipped: set RUN_FULL_PIPELINE=True in the configuration cell.')
else:
    # Strict publication analysis for the added control conditions. This command
    # refuses to report completion unless every expected prompt has all five seeds
    # at Base, SFT, and DPO and all 24 planned tests are present. The frozen primary
    # threshold is 0.82; 0.80 and 0.84 are sensitivity analyses on the same outputs.
    import subprocess, sys
    matched_reports={}
    for threshold in THRESHOLDS:
        threshold_key=str(threshold)
        threshold_output=RUN_PROJECT/"results"/f"matched_control_stagewise_tau_{int(round(threshold*100)):02d}.json"
        cmd=[sys.executable,"-m","experiments.matched_control_stagewise",
             "--project",str(RUN_PROJECT),"--output",str(threshold_output),
             "--threshold",threshold_key,"--seeds",str(SEEDS),"--permutations","10000"]
        if QUICK_TEST:
            cmd.append("--allow-incomplete")
        subprocess.run(cmd,check=True)
        matched_reports[threshold_key]=json.loads(threshold_output.read_text(encoding="utf-8"))
    report=matched_reports[str(EMB_THRESHOLD)]
    (RUN_PROJECT/"results"/"matched_control_stagewise.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    sweep={"status":"complete" if all(r["status"]=="complete" for r in matched_reports.values()) else "incomplete",
           "primary_threshold":EMB_THRESHOLD,"reports":matched_reports}
    (RUN_PROJECT/"results"/"matched_control_threshold_sweep.json").write_text(json.dumps(sweep,indent=2),encoding="utf-8")
    print("Matched-control status:",report["status"],"tests:",report["n_tests"],
          "| threshold sweep:",sweep["status"])
    if report["status"]=="complete":
        subprocess.run([
            sys.executable,"-m","experiments.render_matched_control_results",
            "--report",str(RUN_PROJECT/"results"/"matched_control_stagewise.json"),
            "--markdown",str(RUN_PROJECT/"results"/"matched_control_stagewise.md"),
            "--latex",str(RUN_PROJECT/"results"/"matched_control_stagewise.tex"),
        ],check=True)
        import pandas as pd
        display(pd.DataFrame(report["results"]))
    else:
        print("Incomplete cells (smoke test only):",report["incomplete"][:8])


In [ ]:
#@title A13 · Sync regenerated aggregate artifacts into the repository analysis bundle
if not RUN_FULL_PIPELINE:
    print("Skipped sync: using released artifacts.")
else:
    mapping = {
        "summary_v2.json": "summary_v2.json",
        "interaction_tests.json": "interaction_tests.json",
        "matched_control_stagewise.json": "matched_control_stagewise.json",
        "matched_control_threshold_sweep.json": "matched_control_threshold_sweep.json",
        "matched_control_stagewise.md": "matched_control_stagewise.md",
        "matched_control_stagewise.tex": "matched_control_stagewise.tex",
    }
    for source_name, destination_name in mapping.items():
        source = RUN_PROJECT / "results" / source_name
        destination = REPO_ROOT / "out" / "results" / destination_name
        if not source.exists():
            raise FileNotFoundError(source)
        shutil.copy2(source, destination)
        print("Synced", source, "->", destination)

## B. Corrected paper statistics and robustness

These cells are the default reproducibility path. They apply plus-one Monte
Carlo corrections, prompt-clustered interaction tests, separate BH families,
threshold robustness, and paper-table exports from the released or regenerated
artifact bundle.

In [ ]:
#@title B1 · Regenerate all locally available paper statistics
import subprocess, sys
modules = [
    "experiments.monte_carlo_correction",
    "experiments.derive_paper_stats",
    "experiments.between_condition_stats",
    "experiments.stage_condition_gaps",
    "experiments.multiple_testing",
    "experiments.threshold_robustness",
    "experiments.render_interaction",
]
for module in modules:
    print("\n===", module, "===")
    completed = subprocess.run(
        [sys.executable, "-m", module], cwd=REPO_ROOT,
        check=False, capture_output=True, text=True
    )
    if completed.returncode:
        print(completed.stdout[-2000:])
        print(completed.stderr[-2000:])
        raise RuntimeError(f"{module} failed with exit code {completed.returncode}")
    tail = completed.stdout.strip().splitlines()[-3:]
    print("\n".join(tail) if tail else "completed")
print("All corrected local analyses completed.")

In [ ]:
#@title B2 · Optional remote paired DPO scan (~2.7 GB streamed through DuckDB)
if not RUN_REMOTE_DPO_SCAN:
    path = REPO_ROOT / "out/results/dpo_pairwise.json"
    if not path.exists():
        raise FileNotFoundError("No cached dpo_pairwise.json; enable RUN_REMOTE_DPO_SCAN")
    print("Using released paired DPO artifact:", path)
else:
    subprocess.run([sys.executable, "-m", "experiments.dpo_pairwise"], cwd=REPO_ROOT, check=True)
    subprocess.run([sys.executable, "-m", "experiments.multiple_testing"], cwd=REPO_ROOT, check=True)
    print("Remote paired DPO scan completed and BH results refreshed.")

In [ ]:
#@title B3 · Compact results dashboard
results_dir = REPO_ROOT / "out" / "results"
dpo = json.loads((results_dir / "dpo_pairwise.json").read_text(encoding="utf-8"))
thresholds = json.loads((results_dir / "threshold_robustness.json").read_text(encoding="utf-8"))
testing = json.loads((results_dir / "multiple_testing.json").read_text(encoding="utf-8"))

dpo_rows = []
for model, block in dpo.items():
    for behavior, values in block["behaviors"].items():
        dpo_rows.append({
            "model": model,
            "behavior": behavior,
            "chosen_only": values["chosen_only"],
            "rejected_only": values["rejected_only"],
            "word_rate_ratio": values["token_normalized_ratio"],
            "mcnemar_log10_p": values["mcnemar_log10_p"],
        })
display(pd.DataFrame(dpo_rows))

print("Threshold contrasts:")
threshold_rows = []
for model, by_threshold in thresholds["models"].items():
    for threshold, values in by_threshold.items():
        threshold_rows.append({"model": model, "threshold": threshold, **values})
display(pd.DataFrame(threshold_rows))
print("Multiple-testing families:", list(testing))

## C. Provenance-safe human validation v3

The annotator sees only a stable item ID, prompt (for response-level items), and
text. Detector predictions, model stage, prompt condition, and the other
annotator's file are separate and hidden.

Two distinct humans must each run the initialization and UI cells with their own
`ANNOTATOR_ID`. AI agents may be used only for the separately marked diagnostic
audit; they must never select the human-certification checkbox.

For the current deadline, `sentence_120` reproduces the prepared stratified
sample. For the strongest paper, run the full pipeline and choose
`response_stagewise`, which validates the answer-level measurement across
models, stages, and conditions and adds a detector-positive enriched precision
sample.

In [ ]:
#@title C1 · Prepare blinded items and hidden key
from experiments import validation_v3

if VALIDATION_MODE == "sentence_120":
    manifest = validation_v3.prepare_sentence(
        REPO_ROOT / "out/results/validation_sheet_v2.csv", VALIDATION_DIR
    )
else:
    manifest = validation_v3.prepare_response(
        RUN_PROJECT / "answers",
        RUN_PROJECT / "behavior",
        VALIDATION_DIR,
        random_per_stratum=8,
        positive_per_category=40,
        threshold=0.82,
        seed=0,
    )
print(json.dumps(manifest, indent=2))
print("Blinded items:", VALIDATION_DIR / "validation_items_v3.csv")
print("Hidden scoring key (annotators must not open):", VALIDATION_DIR / "validation_key_v3.csv")

In [ ]:
#@title C2 · Initialize one certified human annotator file
if not I_AM_A_HUMAN_ANNOTATOR:
    print("Not initialized. A real human annotator must set I_AM_A_HUMAN_ANNOTATOR=True and provide ANNOTATOR_NAME.")
elif not ANNOTATOR_NAME.strip():
    raise ValueError("Set a non-empty pseudonymous ANNOTATOR_NAME")
else:
    annotation_path = validation_v3.init_annotation(
        int(ANNOTATOR_ID), ANNOTATOR_NAME.strip(), True, True, VALIDATION_DIR
    )
    print("Human annotation file:", annotation_path)
    print("Work independently. Do not open validation_key_v3.csv or the other annotator's file.")

In [ ]:
#@title C3 · Blinded in-notebook annotation UI (autosaves per item)
import csv
try:
    from IPython.display import display, clear_output
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("Interactive widgets are unavailable in this kernel; run this cell in Colab or edit the separate annotation CSV directly.")

def launch_annotation_ui(annotator: int, directory: Path):
    if widgets is None:
        return
    annotation_path, _ = validation_v3.annotation_paths(annotator, directory)
    if not annotation_path.exists():
        print("Initialize the certified annotation file in C2 first.")
        return
    items = validation_v3.read_csv(directory / "validation_items_v3.csv")
    annotations = validation_v3.read_csv(annotation_path)
    by_id = {row["item_id"]: row for row in annotations}
    state = {"index": 0}
    categories = validation_v3.CATEGORIES
    display_names = {
        "empathy_opener": "empathy expression",
        "validation": "validation",
        "disclaimer": "professional referral",
        "crisis_referral": "crisis referral",
        "structure": "structuring language",
    }
    progress = widgets.HTML()
    item_box = widgets.HTML(layout=widgets.Layout(border="1px solid #888", padding="12px"))
    checks = {category: widgets.Checkbox(description=display_names[category]) for category in categories}
    none_box = widgets.Checkbox(description="none of the above")
    message = widgets.HTML()
    previous = widgets.Button(description="Previous")
    save_next = widgets.Button(description="Save + next", button_style="success")

    def completed_count():
        return sum(bool(row.get("label", "").strip()) for row in annotations)

    def render():
        item = items[state["index"]]
        label = by_id[item["item_id"]].get("label", "")
        selected = validation_v3.parse_label(label)
        for category, checkbox in checks.items():
            checkbox.value = category in selected
        none_box.value = label.strip() == "none"
        prompt = item.get("prompt", "").strip()
        prompt_html = f"<b>Prompt</b><br>{prompt}<hr>" if prompt else ""
        item_box.value = (
            f"<b>Item {state['index'] + 1}/{len(items)}</b> &nbsp; <code>{item['item_id']}</code><br><br>"
            f"{prompt_html}<b>Text to label</b><br><pre style='white-space:pre-wrap'>{item['text']}</pre>"
        )
        progress.value = f"<b>{completed_count()}/{len(items)} labeled</b>"
        message.value = ""

    def save_current():
        chosen = [category for category, checkbox in checks.items() if checkbox.value]
        if none_box.value and chosen:
            raise ValueError("none cannot be combined with another category")
        label = "none" if none_box.value else ",".join(sorted(chosen))
        label = validation_v3.canonical_label(label)
        item_id = items[state["index"]]["item_id"]
        row = by_id[item_id]
        row["label"] = label
        row["completed_at"] = validation_v3.utc_now()
        validation_v3.write_csv(
            annotation_path,
            annotations,
            ["item_id", "label", "annotator_id", "annotator_type", "completed_at"],
        )

    def on_previous(_):
        if state["index"] > 0:
            state["index"] -= 1
        render()

    def on_save_next(_):
        try:
            save_current()
            if state["index"] < len(items) - 1:
                state["index"] += 1
            render()
        except Exception as exc:
            message.value = f"<b style='color:#c33'>{exc}</b>"

    previous.on_click(on_previous)
    save_next.on_click(on_save_next)
    display(progress, item_box, widgets.VBox(list(checks.values()) + [none_box]), widgets.HBox([previous, save_next]), message)
    render()

launch_annotation_ui(int(ANNOTATOR_ID), VALIDATION_DIR)

In [ ]:
#@title C4 · Score two complete human passes and export disagreements
validation_report = validation_v3.score(VALIDATION_DIR)
print("status:", validation_report["status"], "| n:", validation_report.get("n", validation_report.get("n_items")))
if "exact_set_agreement" in validation_report:
    print("exact-set agreement:", validation_report["exact_set_agreement"])
    print("Cohen's kappa:", validation_report["cohen_kappa"])
if "detector_vs_adjudicated_human" in validation_report:
    display(pd.DataFrame(validation_report["detector_vs_adjudicated_human"]).T)
elif "detector_vs_human_1" in validation_report:
    display(pd.DataFrame(validation_report["detector_vs_human_1"]).T)
if validation_report["status"].startswith("pending"):
    print("reason:", validation_report.get("reason", "complete both human files"))
    print("FINAL HUMAN RESULT NOT CREATED. Complete both certified annotation files first.")
elif validation_report["status"].startswith("complete_unadjudicated"):
    print("Both passes complete. Review validation_disagreements_v3.csv, then run C5.")

In [ ]:
#@title C5 · Initialize adjudication and finalize after disagreements are resolved
if not (VALIDATION_DIR / "validation_disagreements_v3.csv").exists():
    print("No disagreement file yet. Complete C4 first.")
else:
    adjudication_path = validation_v3.init_adjudication(VALIDATION_DIR)
    adjudication_rows = validation_v3.read_csv(adjudication_path)
    print("Adjudication file:", adjudication_path)
    print("Fill label and decision_rule for every row, then rerun C4.")
    display(pd.DataFrame(adjudication_rows).head(20))

## D. Optional causal leave-cluster-out SFT pilot

This section is opt-in. It starts from `allenai/OLMo-2-0425-1B` and uses the
**exact model-specific 1B SFT mixture**
`allenai/tulu-3-sft-olmo-2-mixture-0225`, fixing the generic-mixture error in the
older pilot notebook. It compares full subset SFT, behavioral-cluster removal,
coherent distractor removal, and random removal.

The default subset/one-seed configuration is a feasibility pilot, not a final
causal publication claim. A publishable result needs the official recipe,
multiple seeds, matched removal controls, and the frozen human-validated outcome
measure.

In [ ]:
#@title 4 · Config
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    BASE_MODEL   = "allenai/OLMo-2-0425-1B"   # the PRE-SFT base checkpoint
    SFT_SOURCE   = "allenai/tulu-3-sft-olmo-2-mixture-0225"  # exact 1B OLMo-2 mixture
    SUBSET_N     = 20000   #@param {type:"integer"}   # convos sampled from Tulu (tractable)
    METHOD       = "full"  #@param ["full","lora"]      # full-FT = cleaner causal claim; lora = fits T4
    EPOCHS       = 2       #@param {type:"number"}
    LR           = 2e-5    #@param {type:"number"}
    MAXLEN       = 2048    #@param {type:"integer"}
    BATCH        = 2       #@param {type:"integer"}
    GRAD_ACCUM   = 8       #@param {type:"integer"}
    TARGET_BEHAVIOR = "disclaimer"  #@param ["disclaimer","empathy_opener","structure","validation"]
    SEED         = 0       #@param {type:"integer"}
    SAVE_MODELS_TO_DRIVE = False  #@param {type:"boolean"}
    CONDITIONS = ["base_no_sft","full_sft","minus_behavioral","minus_distractor","minus_random"]
    EVAL_PROMPTS = {
     "advice":[
      "I broke up with my girlfriend without any reason and I feel terrible. What should I do?",
      "How do I stop snoring while sleeping with my girlfriend?",
      "I've been feeling really anxious lately and can't sleep. Any advice?",
      "My partner and I keep fighting about small things. How can we communicate better?",
      "I feel lonely all the time even when I'm around people. What can I do?",
      "I think I'm having panic attacks. What should I do when one happens?",
      "I'm really stressed about work and it's affecting my health. What can I do?",
      "My friend is going through a hard time and I don't know how to support them."],
     "factual":[
      "What is the capital of Australia?","How does photosynthesis work?",
      "What causes the seasons on Earth?","Explain how a bicycle gear system works."],
    }
    print(f"{BASE_MODEL} | N={SUBSET_N} | {METHOD} | target={TARGET_BEHAVIOR} | seed={SEED}")


In [ ]:
#@title 5 · Checkpoint helpers + behavior detectors (lexicon + embedding)
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    import json, re, numpy as np
    def ck(stage,key): return CAUSAL_PROJECT/stage/f"{key}.json"
    def have(stage,key): return ck(stage,key).exists()
    def load_ck(stage,key): return json.loads(ck(stage,key).read_text(encoding='utf-8'))
    def save_ck(stage,key,o): ck(stage,key).write_text(json.dumps(o,indent=2,ensure_ascii=False),encoding='utf-8'); return o

    LEXICON={
     "empathy_opener":["i'm sorry to hear","i am sorry to hear","i'm really sorry","that sounds really","that sounds very","i can understand","i can imagine","it's understandable","that must be","i'm here for you","i hear you","it sounds like you"],
     "validation":["your feelings are valid","it's okay to feel","it's normal to feel","it's completely normal","what you're feeling","there's nothing wrong with","it's natural to"],
     "disclaimer":["consult a","talk to a","speak with a","see a doctor","seek medical","i am not a doctor","i'm not a doctor","medical professional","healthcare professional","mental health professional","seek professional","professional help"],
     "crisis_referral":["crisis","hotline","helpline","988","call 911","emergency services","suicide"],
     "structure":["here are some","here are a few","consider the following","steps you can take","a few things you can","first,","secondly,"],
    }
    ALL_PHRASES=[(c,p) for c,ps in LEXICON.items() for p in ps]
    def find_behaviors(t):
        low=t.lower(); out=[]
        for c,p in ALL_PHRASES:
            i=low.find(p)
            while i!=-1: out.append((c,p,i)); i=low.find(p,i+1)
        return out
    def lex_categories(t): return sorted({c for c,_,_ in find_behaviors(t)})

    from sentence_transformers import SentenceTransformer
    _emb=SentenceTransformer("intfloat/multilingual-e5-small")
    def enc(ts,pfx): return _emb.encode([f"{pfx}: {x}" for x in ts],normalize_embeddings=True,batch_size=256,show_progress_bar=False)
    EXEMPLARS={
     "empathy_opener":["I'm so sorry you're going through this.","That sounds incredibly hard."],
     "validation":["Your feelings are completely valid.","It's totally normal to feel this way."],
     "disclaimer":["You should consult a qualified professional.","Please talk to your doctor about this."],
     "crisis_referral":["If you're in crisis, please call a helpline.","Contact a suicide prevention hotline."],
     "structure":["Here are a few steps you can take.","Consider the following suggestions."],
    }
    EXV={c:enc(v,"passage") for c,v in EXEMPLARS.items()}
    def _sents(t): return [s.strip() for s in re.split(r"(?<=[.!?])\s+",t.strip()) if len(s.strip())>3]
    def emb_categories(t,thr=0.82):
        s=_sents(t)
        if not s: return {}
        sv=enc(s,"query"); out={}
        for c,ev in EXV.items():
            h=int(((sv@ev.T).max(axis=1)>=thr).sum())
            if h: out[c]=h
        return out
    def categories(t): return set(lex_categories(t))|set(emb_categories(t).keys())


In [ ]:
#@title 6 · STAGE 0 — Load Tulu subset + build selector clusters (checkpointed)
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    import random
    from datasets import load_dataset
    random.seed(SEED)
    if have('clusters','manifest'):
        man=load_ck('clusters','manifest'); print('clusters cached:',{k:len(v) for k,v in man.items() if isinstance(v,list)})
    else:
        print('streaming Tulu subset…')
        ds=load_dataset(SFT_SOURCE,split='train',streaming=True,token=HF_TOKEN)
        convos=[]
        for ex in ds:
            convos.append(ex['messages']);
            if len(convos)>=SUBSET_N: break
        def asst_text(m): return " ".join(t.get('content','') for t in m if t.get('role')=='assistant')
        asst=[asst_text(m) for m in convos]
        # auditable behavioral selector: assistant turns matching TARGET_BEHAVIOR phrases
        beh_ids=[i for i,a in enumerate(asst) if any(p in a.lower() for p in LEXICON[TARGET_BEHAVIOR])]
        k=len(beh_ids); print('behavioral cluster size:',k)
        # coherent distractor: KMeans on assistant embeddings, pick a non-behavioral cluster, size-matched
        from sklearn.cluster import KMeans
        sample_idx=list(range(len(asst)))
        V=enc([asst[i][:500] for i in sample_idx],"passage")
        km=KMeans(n_clusters=20,n_init=4,random_state=SEED).fit(V)
        labels=km.labels_; beh_set=set(beh_ids)
        from collections import Counter
        # cluster with fewest behavioral members and >=k items = coherent distractor
        cl_counts={c:[i for i in sample_idx if labels[i]==c] for c in range(20)}
        cand=sorted(cl_counts.values(),key=lambda ids:(sum(i in beh_set for i in ids), -len(ids)))
        # primarily the cleanest coherent cluster; pad from next-cleanest if short, to size-match k
        distractor=[]; ci=0
        while len(distractor)<k and ci<len(cand):
            distractor+=[i for i in cand[ci] if i not in beh_set and i not in distractor]; ci+=1
        distractor=distractor[:k]
        pool=[i for i in range(len(asst)) if i not in beh_set]
        random.shuffle(pool); rand_ids=pool[:k]
        if k<20: print(f"WARNING: behavioral cluster is small (k={k}); raise SUBSET_N or pick a more common TARGET_BEHAVIOR for a detectable effect.")
        man={"n":len(convos),"target":TARGET_BEHAVIOR,
             "behavioral":beh_ids,"distractor":distractor,"random":rand_ids}
        save_ck('clusters','manifest',man)
        # persist the convos to local disk for training (not Drive)
        import pickle
        with open('/content/convos.pkl','wb') as f: pickle.dump(convos,f)
        print('clusters:',{k2:len(v) for k2,v in man.items() if isinstance(v,list)})


In [ ]:
#@title 7 · Training + eval functions
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    import torch, pickle, gc, dataclasses, inspect
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer, SFTConfig
    tok=AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.chat_template is None:   # base OLMo-2 has NO chat template; Instruct sibling shares the vocab
        print("base tokenizer has no chat_template; loading", BASE_MODEL+"-Instruct", "template")
        tok=AutoTokenizer.from_pretrained(BASE_MODEL+"-Instruct")
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    # bf16 only on Ampere+ (A100/L4); T4 is fp16 — auto-detect so it runs anywhere.
    BF16OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    DTYPE  = torch.bfloat16 if BF16OK else torch.float16
    print("precision:", "bf16" if BF16OK else "fp16")

    def _load_convos():
        try:
            with open('/content/convos.pkl','rb') as f: return pickle.load(f)
        except FileNotFoundError:
            from datasets import load_dataset
            ds=load_dataset(SFT_SOURCE,split='train',streaming=True,token=HF_TOKEN); c=[]
            for ex in ds:
                c.append(ex['messages'])
                if len(c)>=SUBSET_N: break
            with open('/content/convos.pkl','wb') as f: pickle.dump(c,f)
            return c

    def build_text_dataset(remove_ids):
        convos=_load_convos(); rm=set(remove_ids)
        from datasets import Dataset
        texts=[tok.apply_chat_template(m,tokenize=False) for i,m in enumerate(convos) if i not in rm]
        return Dataset.from_dict({"text":texts})

    def train_condition(cond, remove_ids):
        # No device_map on a trainable model (fragile with Trainer); let Trainer place it on GPU.
        model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,torch_dtype=DTYPE)
        if METHOD=="lora":
            from peft import LoraConfig, get_peft_model
            model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj"],task_type="CAUSAL_LM"))
        dset=build_text_dataset(remove_ids)
        out=f"/content/sft_{cond}"
        # TRL renamed max_seq_length -> max_length (>=0.20). Build kwargs by introspection so
        # the SAME notebook works across TRL versions.
        fields={f.name for f in dataclasses.fields(SFTConfig)}
        kw=dict(output_dir=out,num_train_epochs=EPOCHS,per_device_train_batch_size=BATCH,
                gradient_accumulation_steps=GRAD_ACCUM,learning_rate=LR,
                bf16=BF16OK,fp16=not BF16OK,logging_steps=25,save_strategy="no",report_to=[],seed=SEED)
        if "dataset_text_field" in fields: kw["dataset_text_field"]="text"
        kw["max_length" if "max_length" in fields else "max_seq_length"]=MAXLEN
        cfg=SFTConfig(**kw)
        # SFTTrainer renamed tokenizer -> processing_class; pass whichever it accepts.
        sig=inspect.signature(SFTTrainer.__init__).parameters
        tkw={"processing_class":tok} if "processing_class" in sig else ({"tokenizer":tok} if "tokenizer" in sig else {})
        tr=SFTTrainer(model=model,args=cfg,train_dataset=dset,**tkw)
        tr.train()
        if SAVE_MODELS_TO_DRIVE: tr.save_model(str(CAUSAL_PROJECT/'models'/cond))
        return model

    @torch.no_grad()
    def gen(model,q):
        enc_=tok.apply_chat_template([{"role":"user","content":q}],add_generation_prompt=True,
                return_tensors="pt",return_dict=True).to(model.device)
        o=model.generate(**enc_,max_new_tokens=256,do_sample=True,temperature=0.7,top_p=0.9,
                         pad_token_id=tok.eos_token_id)
        return tok.decode(o[0][enc_["input_ids"].shape[-1]:],skip_special_tokens=True).strip()

    def eval_model(model):
        rows=[]
        for tag,ps in EVAL_PROMPTS.items():
            for i,q in enumerate(ps):
                a=gen(model,q); rows.append({"tag":tag,"q":q,"answer":a,"cats":sorted(categories(a))})
        cats=list(LEXICON)
        adv=[r for r in rows if r["tag"]=="advice"]
        emis={c:round(sum(c in r["cats"] for r in adv)/max(1,len(adv)),3) for c in cats}
        return {"emission_advice":emis,"answers":rows}

    def free(model):
        del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
#@title 8 · STAGE 1 — Run all conditions (checkpointed; resumes on rerun)
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    man=load_ck('clusters','manifest')
    REMOVE={"full_sft":[],"minus_behavioral":man["behavioral"],
            "minus_distractor":man["distractor"],"minus_random":man["random"]}
    for cond in CONDITIONS:
        if have('eval',cond): print(cond,'cached'); continue
        print('=== condition:',cond,'===',flush=True)
        if cond=="base_no_sft":
            m=AutoModelForCausalLM.from_pretrained(BASE_MODEL,torch_dtype=DTYPE,device_map="auto")
        else:
            m=train_condition(cond,REMOVE[cond])
        ev=eval_model(m); ev["condition"]=cond; ev["n_removed"]=len(REMOVE.get(cond,[]))
        save_ck('eval',cond,ev); free(m); print(cond,'emission:',ev["emission_advice"])
    print('Stage 1 done.')


In [ ]:
#@title 9 · STAGE 2 — Causal table + plot
if not RUN_CAUSAL_PILOT:
    print('Skipped: set RUN_CAUSAL_PILOT=True in the configuration cell.')
else:
    import matplotlib.pyplot as plt, numpy as np
    evals={c:load_ck('eval',c) for c in CONDITIONS if have('eval',c)}
    tb=TARGET_BEHAVIOR
    base=evals.get("full_sft",{}).get("emission_advice",{}).get(tb,0)
    def drop(c): return round(base-evals[c]["emission_advice"].get(tb,0),3) if c in evals else None
    result={"target":tb,"full_sft_emission":base,
            "drop_minus_behavioral":drop("minus_behavioral"),
            "drop_minus_distractor":drop("minus_distractor"),
            "drop_minus_random":drop("minus_random"),
            "base_no_sft_emission":evals.get("base_no_sft",{}).get("emission_advice",{}).get(tb)}
    causal = (result["drop_minus_behavioral"] or 0) > max(result["drop_minus_distractor"] or 0, result["drop_minus_random"] or 0)
    result["causal_signal"]=bool(causal)
    save_ck('results','causal_summary',result)
    print(json.dumps(result,indent=2))
    print('\\nCAUSAL SIGNAL (targeted drop > control drops):', causal)

    cats=list(LEXICON); conds=[c for c in CONDITIONS if c in evals]
    M=np.array([[evals[c]["emission_advice"][k] for k in cats] for c in conds])
    plt.figure(figsize=(10,4)); im=plt.imshow(M,aspect='auto',cmap='Greens',vmin=0,vmax=1)
    plt.xticks(range(len(cats)),cats,rotation=30,ha='right'); plt.yticks(range(len(conds)),conds)
    plt.colorbar(im,label='advice emission rate'); plt.title(f'Behavior emission by condition (target={tb})')
    for i in range(len(conds)):
        for j in range(len(cats)): plt.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=8)
    plt.tight_layout(); plt.savefig(CAUSAL_PROJECT/'results'/'causal_heatmap.png',dpi=130); plt.show()


In [ ]:
#@title E · Final status and export bundle
from zipfile import ZipFile, ZIP_DEFLATED

status_path = VALIDATION_DIR / "validation_status_v3.json"
validation_status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {"status": "not_started"}
final_status = {
    "artifact_statistics": "complete",
    "paired_dpo": "complete" if (REPO_ROOT / "out/results/dpo_pairwise.json").exists() else "missing",
    "matched_control_stagewise": (
        json.loads((REPO_ROOT / "out/results/matched_control_stagewise.json").read_text(encoding="utf-8")).get("status")
        if (REPO_ROOT / "out/results/matched_control_stagewise.json").exists() else "not_run"
    ),
    "human_validation": validation_status.get("status", "not_started"),
    "causal_ablation": "pilot_completed" if RUN_CAUSAL_PILOT and (CAUSAL_PROJECT / "results").exists() else "not_run",
    "submission_rule": "Do not claim completed human validation unless status is complete_adjudicated_two_human_validation.",
}
print(json.dumps(final_status, indent=2))

bundle = RUN_PROJECT / "blackboxnlp_results_bundle.zip"
with ZipFile(bundle, "w", ZIP_DEFLATED) as archive:
    for path in sorted((REPO_ROOT / "out/results").glob("*.json")):
        archive.write(path, arcname=f"paper_results/{path.name}")
    for path in sorted(list(VALIDATION_DIR.glob("*.json")) + list(VALIDATION_DIR.glob("*.csv"))):
        archive.write(path, arcname=f"validation_v3/{path.name}")
print("Export bundle:", bundle)